# JADCO / Collection Équinoxe — soumission du backup 2026

**Méthode : Fixed 50/50 Backup Ensemble — Model A + Model B P1_G1**  
**Résultat portefeuille 2026 : calculé par les pipelines ci-dessous, jamais saisi manuellement.**

Ce notebook est le livrable principal. Il appelle les APIs finales existantes, ne réentraîne pas les modèles et ne change ni les cutoffs, ni les cohortes, ni les configurations gelées. Il affiche uniquement des résultats agrégés. Les données CRM restent locales et ne doivent jamais être ajoutées au dépôt ou aux livrables.

## 1. Challenge / Business Question

Estimer l'indice demandé pour les unités dont le bail vient à échéance, à partir des seules informations admissibles à l'origine de prédiction. Le résultat est exprimé en pourcentage et décrit une contribution attendue des transitions au niveau de la cohorte; ce n'est **pas** la croissance totale du rent roll, du revenu, du NOI ou de tous les loyers.

Ce dépôt contient les pipelines et le notebook, mais exclut les CSV CRM confidentiels. Pour reproduire les calculs, placer localement `equinoxe_lease_history.csv` dans `data/raw/`. Aucun chemin absolu propre à une machine n'est utilisé.

## 2. Data Understanding

L'entrée du modèle est l'historique des baux. Le dashboard accepte aussi listings, concessions et asking history pour compatibilité; ils ne sont pas des entrées prédictives des pipelines finaux. Les identifiants sont chargés comme texte. Cette cellule ne montre ni lignes CRM, ni identifiants, ni valeurs de colonnes.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
project_root = next(
    (candidate for candidate in (current, *current.parents)
     if (candidate / "src" / "models" / "ensemble_50_50.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Run this notebook from within the project checkout.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

lease_path = project_root / "data" / "raw" / "equinoxe_lease_history.csv"
if not lease_path.is_file():
    raise FileNotFoundError(
        "Confidential input is not included in Git. Place the authorized "
        "equinoxe_lease_history.csv locally at data/raw/equinoxe_lease_history.csv."
    )

leases = pd.read_csv(
    lease_path,
    dtype={
        "sPropCode": "string",
        "sUnitCode": "string",
        "sBuilding": "string",
        "sState": "string",
    },
)
dataset_summary = pd.DataFrame(
    [{"dataset": "Lease history", "rows": len(leases), "columns": len(leases.columns)}]
)
display(dataset_summary)

## 3. Definition of Target

Cible : **« Indice médian de contribution attendue des transitions à la croissance annualisée du loyer effectif des unités à échéance »**.

- `p_i` = probabilité de transition de l'unité candidate `i` ;
- `g_i` = croissance annualisée attendue du loyer effectif conditionnelle à une transition ;
- `c_i = p_i * g_i` = contribution attendue ;
- `P1 = median(c_i)` sur la cohorte de candidats.

Les composantes sont calculées comme fractions dans les pipelines; l'API rapporte `P1` en pourcentage. Une médiane des contributions événementielles n'est pas une croissance complète du rent roll.

## 4. Composition Effect / Mix

Une moyenne ou médiane de loyers agrégés peut bouger lorsque le portefeuille change de composition, même sans hausse same-unit comparable. Par exemple, l'ajout d'immeubles haut de gamme peut relever le loyer moyen apparent. L'appariement de chaque unité à son bail précédent réduit cet effet de niveau des nouvelles unités; il ne fixe toutefois pas la composition des transitions ni le mix renouvellement/relocation de la cohorte. Le P1 agrégé n'est donc pas un indice à mix constant.

## 5. Same-Unit Method

La clé officielle est `sPropCode + sUnitCode` (`UNIT_KEY` dans la foundation). **Ne pas** substituer `sSite + sUnitCode`. Les transitions comparent une unité à son bail précédent admissible; la construction, l'ordre et les règles de validité sont délégués à la foundation gelée. Les identifiants servent uniquement à l'appariement en mémoire et ne sont pas exportés.

## 6. Contractual vs Effective Rent / Concessions

`sRent` est le loyer contractuel; `sRentEffective` est le loyer effectif fourni dans la source. La cible principale utilise le loyer effectif, sans reconstruire celui-ci à partir du loyer contractuel. Concessions, mois gratuits et autres avantages peuvent faire diverger la croissance contractuelle de la croissance économique effective. Le loyer contractuel demeure distinct et ne doit pas être présenté comme un substitut équivalent.

## 7. Renewals vs Relocations

Un renouvellement et une relocation/turnover peuvent suivre des dynamiques de prix et de concessions différentes. Leur distinction a été étudiée dans les diagnostics historiques. Le statut du nouveau bail (`sRenewal`) n'est connu qu'après l'origine de prédiction : il ne sert donc pas de feature de forecast. Le mix futur des renouvellements et relocations peut néanmoins changer et affecter la médiane observée.

## 8. Québec vs Ontario

Le portefeuille comprend cinq immeubles au Québec et **The Met** en Ontario. Les paramètres du TAL concernent le Québec; la guideline d'augmentation ontarienne concerne l'Ontario et son applicabilité à chaque logement doit être vérifiée. Aucune règle québécoise n'est appliquée automatiquement à l'Ontario, ni inversement. Les sources externes ne sont pas des features du backup.

## 9. External Data

Sources publiques étudiées : SCHL/CMHC (marchés locatifs), Statistique Canada (IPC des loyers, table 18-10-0004-01), Tribunal administratif du logement du Québec (TAL) et guideline d'augmentation des loyers de l'Ontario. Les univers mesurés (loyers moyens, IPC, règles ou marché locatif) ne sont pas identiques à la croissance same-unit effective de JADCO.

L'admissibilité point-in-time exige `availability_verified == True` et `available_date <= cutoff`; `retrieved_date` n'est jamais un substitut. Selon l'audit des sources du dépôt, seules neuf composantes TAL de 2025 sont vérifiées disponibles avant le cutoff 2025-12-31; les lignes SCHL, Statistique Canada et Ontario ne sont pas historiquement qualifiées dans le fichier. Aucune donnée externe n'est intégrée quantitativement aux modèles/backtests de cet ensemble; les signaux restent du contexte/sensibilité plutôt que des features rétroactivement antidatées.

## 10. Leakage-Safe Forecasting Protocol

Origines inclusives gelées : 2023 → 2022-12-31; 2024 → 2023-12-31; 2025 → 2024-12-31; forecast 2026 → 2025-12-31. Les cohortes, caractéristiques et labels d'entraînement sont construits par les fonctions existantes à chaque cutoff. Les cibles de l'année test ne sont révélées qu'après les prédictions et ne servent qu'à évaluer. Aucune information après chaque origine n'est autorisée. La fidélité des dates historiques du CRM demeure une limite documentée, car il ne s'agit pas d'un journal versionné.

## 11. Model A

Model A est le modèle global conservateur gelé par l'équipe : il utilise ses estimateurs historiques globaux/médians et sa cohorte conforme à la foundation. Son API publique `backtest(leases, target_year)` calcule les folds et `estimate_2026(leases, asking, external=None)` produit son estimation finale. Le wrapper d'ensemble appelle ces APIs; ce notebook ne recopie pas la logique Model A et ne la modifie pas.

## 12. Model B P1_G1

Model B P1_G1 utilise la configuration finalisée : occurrence hiérarchique province → province + mois d'échéance, shrinkage 20; croissance conditionnelle hiérarchique par province, shrinkage 1. Le tournoi historique existant fournit directement P1_G1. Pour 2026, le wrapper applique les mêmes estimateurs et configurations gelés au cutoff 2025-12-31. Aucun autre réglage, feature externe ou résultat 2026 observé n'est utilisé.

## 13. Fixed 50/50 Backup Ensemble

Nom de méthode : **Fixed 50/50 Backup Ensemble (Model A + Model B P1_G1)**.

`Forecast_backup = 0.50 * Forecast_A + 0.50 * Forecast_B_P1_G1`.

**The 50/50 weights are fixed ex ante and were not optimized on the historical backtests.** Le notebook appelle les wrappers qui réutilisent les APIs finales; aucune logique de modèle n'est dupliquée.

In [ ]:
from src.models.ensemble_50_50 import (
    ENSEMBLE_WEIGHTS,
    backtest_ensemble_50_50,
    estimate_2026_ensemble,
)

assert dict(ENSEMBLE_WEIGHTS) == {"model_a": 0.5, "model_b": 0.5}
backtest_rows, comparison_metrics = backtest_ensemble_50_50(leases)
forecast = estimate_2026_ensemble(leases, asking=None)
expected_ensemble = (
    ENSEMBLE_WEIGHTS["model_a"] * forecast["model_a"]
    + ENSEMBLE_WEIGHTS["model_b"] * forecast["model_b"]
)
assert forecast["ensemble_50_50"] == expected_ensemble
assert forecast["prediction_origin"] == "2025-12-31"
assert backtest_rows["year"].tolist() == [2023, 2024, 2025]
print("Fixed weights:", dict(ENSEMBLE_WEIGHTS))
print("2026 prediction origin:", forecast["prediction_origin"])
print("Model B candidate count:", forecast["model_b_candidate_count"])

## 14. Backtests 2023–2025

Le wrapper réutilise `backtest(leases, target_year)` de Model A et les résultats du tournoi P1_G1 de Model B. Il vérifie l'origine, le nombre de candidats et la cible réalisée avant comparaison. Les erreurs sont en points de pourcentage; le biais positif indique une surestimation. Les années reçoivent un poids égal dans les métriques.

In [ ]:
display(backtest_rows.round(6))
display(comparison_metrics.round(6))
chart = backtest_rows.set_index("year")[[
    "realized", "A_prediction", "B_prediction", "ensemble_50_50"
]].rename(columns={
    "realized": "Actual",
    "A_prediction": "Model A",
    "B_prediction": "Model B P1_G1",
    "ensemble_50_50": "Ensemble 50/50",
})
chart.plot(marker="o", title="Historical P1 backtests (percent)", ylabel="P1 (%)")

## 15. Final 2026 Forecast

Les valeurs sont calculées ci-dessous par le wrapper au cutoff 2025-12-31. Aucun résultat ou label 2026 n'est consulté. Le chiffre ensemble est exactement la moyenne arithmétique à poids fixes.

In [ ]:
forecast_summary = pd.DataFrame([
    {"model": "Model A 2026", "forecast_percent": forecast["model_a"]},
    {"model": "Model B P1_G1 2026", "forecast_percent": forecast["model_b"]},
    {"model": "FINAL BACKUP ENSEMBLE 50/50", "forecast_percent": forecast["ensemble_50_50"]},
])
display(forecast_summary.assign(
    forecast_percent=forecast_summary["forecast_percent"].map(lambda value: f"{value:.6f} %")
))
print("Prediction origin:", forecast["prediction_origin"])
print("Exact arithmetic check:", forecast["ensemble_50_50"] == expected_ensemble)

## 16. Limitations

- Seulement trois années de backtest (2023–2025); les erreurs ne donnent pas une estimation robuste de toute la distribution future.
- Le taux de transition a dérivé matériellement en 2025; il peut affecter la stabilité des relations historiques.
- P1 est un indice événementiel médian de contribution attendue, pas la croissance complète du rent roll.
- Les signaux externes ont une disponibilité historique vérifiée limitée et ne sont pas intégrés quantitativement à cet ensemble.
- Model A et Model B reposent sur des hypothèses différentes; l'ensemble fixe ne supprime pas l'incertitude de modèle.
- L'ensemble 50/50 est un fallback neutre à poids fixes, pas un ensemble optimisé.
- La fidélité des dates et révisions du CRM historique n'est pas entièrement démontrée sans journal de versions.
- Les données CRM sont confidentielles : elles restent locales et ne doivent pas être téléversées dans un dépôt public.

## 17. References / AI Tools

**Sources publiques**
- [SCHL / CMHC — Housing market information](https://www.cmhc-schl.gc.ca/)
- [Statistique Canada — Table 18-10-0004-01](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1810000401)
- [Tribunal administratif du logement du Québec](https://www.tal.gouv.qc.ca/)
- [Ontario rent increase guideline](https://www.ontario.ca/page/rent-increase-guideline)

La disponibilité et les limites d'usage des séries retenues sont consignées dans `docs/external_sources.md`; la documentation du dépôt et la foundation gelée définissent le protocole interne. Les références externes contextualisent le marché et ne redéfinissent pas la cible.

**AI tools used:** ChatGPT / OpenAI; GitHub Copilot. Les outils d'IA ont aidé au développement et à la documentation; les calculs présentés sont exécutés par les fonctions du dépôt.